<a href="https://www.kaggle.com/code/marrioqqqq/01-hyperparameter-optimization-tpe?scriptVersionId=336960293" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install -q optuna optuna-integration[tfkeras]

import os
import random
from enum import Enum
from typing import Tuple, List, Dict, Any

import numpy as np
import tensorflow as tf

class ModelArch(str, Enum):
    MOBILENET_V3 = 'MobileNetV3Small'
    EFFICIENTNET_B0 = 'EfficientNetB0'
    RESNET_50 = 'ResNet50'

class HPOConfig:    
    # Konfigurasi Dataset & Dimensi
    PATH_TRAIN_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/train"
    PATH_VAL_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-sawit/dataset-sawit/val"
    IMG_SIZE: Tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 3
    
    # # Konfigurasi Optuna The Grand Loop
    # N_TRIALS: int = 50
    # EPOCHS_HPO: int = 50
    # N_STARTUP_TRIALS: int = 5 # Fase explorasi sebelum Pruning Optuna aktif
    # N_WARMUP_STEPS: int = 10  # Epoch minimal sebelum trial boleh dihentikan (Pruned)

    # Konfigurasi Optuna The Grand Loop (MODE DEBUG)
    N_TRIALS: int = 2        # Hanya lakukan 2 percobaan per model
    EPOCHS_HPO: int = 1      # Hanya latih 1 epoch per trial
    N_STARTUP_TRIALS: int = 0 
    N_WARMUP_STEPS: int = 0
    
    # Search Space Boundaries (Batas Ekstrem Eksperimen)
    SEARCH_SPACE = {
        'batch_size': [8, 16, 32],
        'learning_rate': (1e-4, 1e-2),
        'weight_decay': (1e-5, 1e-2),
        'dropout_rate': (0.1, 0.3, 0.1), # min, max, step
        'optimizer': ['adamw', 'adam', 'sgd']
    }
    
    # Direktori Output Dinamis
    BASE_OUTPUT_DIR: str = "/kaggle/working/hpo_outputs"

    @classmethod
    def get_output_dir(cls, arch: ModelArch) -> str:
        """Standard 4: Men-generate folder spesifik per arsitektur untuk mencegah overwrite."""
        target_dir = os.path.join(cls.BASE_OUTPUT_DIR, arch.value)
        os.makedirs(target_dir, exist_ok=True)
        return target_dir

    @classmethod
    def get_db_path(cls, arch: ModelArch) -> str:
        return os.path.join(cls.get_output_dir(arch), f"study_{arch.value}.db")

    @classmethod
    def get_model_save_path(cls, arch: ModelArch) -> str:
        # Standard 7: Ekstensi wajib .keras
        return os.path.join(cls.get_output_dir(arch), f"best_{arch.value}.keras")

# ------------------------------------------------------------------------------
# 3. REPRODUKSIBILITAS (Standard 3)
# ------------------------------------------------------------------------------
def set_deterministic_seed(seed: int = 42) -> None:
    """Mengunci semua seed generator untuk menjamin hasil bisa direplikasi."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    print(f"[SETUP] Deterministic Seed dikunci pada: {seed}")

# ------------------------------------------------------------------------------
# 4. INISIALISASI LINGKUNGAN
# ------------------------------------------------------------------------------
set_deterministic_seed(42)

# Mengaktifkan Mixed Precision (Opsional: Mempercepat proses di GPU T4/P100)
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("[SETUP] Mixed Precision (FP16) Aktif.")
except Exception as e:
    print(f"[WARNING] Gagal mengaktifkan Mixed Precision: {e}")

print(f"[SETUP] TensorFlow Version: {tf.__version__}")
print(f"[SETUP] GPU Ditemukan: {len(tf.config.list_physical_devices('GPU'))}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 4.8 MB/s eta 0:00:00
[SETUP] Deterministic Seed dikunci pada: 42
[SETUP] Mixed Precision (FP16) Aktif.
[SETUP] TensorFlow Version: 2.20.0
[SETUP] GPU Ditemukan: 1


In [2]:
# ==============================================================================
# [CELL 2] BASE CLASS: FUNCTIONAL GRAPH BUILDER & UNIT TESTING
# ==============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, applications

class FFBGraphBuilder:
    """
    Standard 9: Base Class untuk membangun arsitektur model secara dinamis
    dan menjalankan Unit Testing sebelum pelatihan berat dimulai.
    """
    
    def __init__(self, config: HPOConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.model_save_path = self.cfg.get_model_save_path(self.arch)
        
        print(f"\n" + "="*50)
        print(f"🛠️ INISIALISASI BUILDER: {self.arch.value}")
        print("="*50)

    # --------------------------------------------------------------------------
    # MODUL BUILDER: FUNCTIONAL API (Standard 7)
    # --------------------------------------------------------------------------
    def _build_functional_graph(self, dropout_rate: float) -> models.Model:
        """Membangun graf model secara eksplisit menggunakan Functional API."""
        inputs = tf.keras.Input(shape=(self.cfg.IMG_SIZE[0], self.cfg.IMG_SIZE[1], 3), name="input_ffb")
        
        # Augmentasi Layer (Dibungkus dalam Sequential agar mudah dicabut saat PTQ)
        augmentation_pipeline = tf.keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.3),
            layers.RandomTranslation(0.1, 0.1),
            layers.RandomZoom(0.2),
            layers.RandomBrightness(0.2),
            layers.RandomContrast(0.2),
        ], name="augmentation_block")
        
        x = augmentation_pipeline(inputs)

        # Dynamic Backbone Loading (Standard 5)
        if self.arch == ModelArch.MOBILENET_V3:
            backbone = applications.MobileNetV3Small(input_tensor=x, include_top=False, weights='imagenet')
        elif self.arch == ModelArch.EFFICIENTNET_B0:
            backbone = applications.EfficientNetB0(input_tensor=x, include_top=False, weights='imagenet')
        elif self.arch == ModelArch.RESNET_50:
            x = layers.Lambda(applications.resnet50.preprocess_input, name="resnet_preprocess")(x)
            backbone = applications.ResNet50(input_tensor=x, include_top=False, weights='imagenet')
        else:
            raise ValueError("Arsitektur tidak dikenali!")
        
        # Bekukan backbone untuk fase HPO / Transfer Learning awal
        backbone.trainable = False
        x = backbone.output

        # Classifier Head
        x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
        x = layers.BatchNormalization(name="head_bn")(x)
        x = layers.Dropout(dropout_rate, name="head_dropout")(x)
        
        # Wajib dtype='float32' untuk menstabilkan Mixed Precision (Standard 7)
        outputs = layers.Dense(self.cfg.NUM_CLASSES, activation='softmax', 
                               name=f"{self.arch.value}_output", dtype='float32')(x)

        return models.Model(inputs=inputs, outputs=outputs, name=f"Edge_{self.arch.value}")

    # --------------------------------------------------------------------------
    # UNIT TESTING (Standard 9)
    # --------------------------------------------------------------------------
    def run_unit_test(self) -> None:
        """Memvalidasi integritas graf komputasi menggunakan dummy tensor."""
        print("[TEST] Menjalankan Sanity Check arsitektur...")
        try:
            test_model = self._build_functional_graph(dropout_rate=0.2)
            dummy_tensor = tf.random.normal((1, self.cfg.IMG_SIZE[0], self.cfg.IMG_SIZE[1], 3))
            dummy_output = test_model(dummy_tensor, training=False)
            
            assert dummy_output.shape == (1, self.cfg.NUM_CLASSES), "Bentuk output classifier salah!"
            assert dummy_output.dtype == tf.float32, "Tipe data output harus Float32!"
            print(f"[TEST] LULUS! Output Shape: {dummy_output.shape} | Parameter: {test_model.count_params():,}")
            
            del test_model, dummy_tensor
            tf.keras.backend.clear_session()
        except Exception as e:
            raise RuntimeError(f"[TEST] GAGAL: Unit test arsitektur menemukan error: {e}")

print("[BERHASIL] Cell 2 (Base Class Arsitektur) dimuat.")

[BERHASIL] Cell 2 (Base Class Arsitektur) dimuat.


In [3]:
# ==============================================================================
# [CELL 3] CORE ENGINE: OPTUNA OBJECTIVE & BEST PARAMS EXPORTER
# ==============================================================================
import json
import optuna
from optuna.integration import TFKerasPruningCallback
from tensorflow.keras import optimizers, callbacks

class FFBHPOEngine(FFBGraphBuilder):
    """
    Mewarisi fungsionalitas graf dari FFBGraphBuilder. 
    Khusus menangani pencarian Hyperparameter Optuna dan ekspor konfigurasi.
    """
    
    def __init__(self, config: HPOConfig, architecture: ModelArch):
        super().__init__(config, architecture)
        
        self.study_name = f"FFB_{self.arch.value}_HPO"
        self.db_url = f"sqlite:///{self.cfg.get_db_path(self.arch)}"
        
        # Standard 4: File JSON untuk menyimpan parameter terbaik
        self.best_params_path = os.path.join(self.cfg.get_output_dir(self.arch), f"best_params_{self.arch.value}.json")

    # --------------------------------------------------------------------------
    # FUNGSI OBJEKTIF OPTUNA
    # --------------------------------------------------------------------------
    def _objective(self, trial: optuna.trial.Trial) -> float:
        tf.keras.backend.clear_session() 

        # 1. Sugesti Hyperparameter
        bs = trial.suggest_categorical('batch_size', self.cfg.SEARCH_SPACE['batch_size'])
        
        lr_cfg = self.cfg.SEARCH_SPACE['learning_rate']
        lr = trial.suggest_float('learning_rate', lr_cfg[0], lr_cfg[1], log=True)
        
        wd_cfg = self.cfg.SEARCH_SPACE['weight_decay']
        wd = trial.suggest_float('weight_decay', wd_cfg[0], wd_cfg[1], log=True)
        
        opt_str = trial.suggest_categorical('optimizer', self.cfg.SEARCH_SPACE['optimizer'])
        
        drop_cfg = self.cfg.SEARCH_SPACE['dropout_rate']
        drop = trial.suggest_float('dropout_rate', drop_cfg[0], drop_cfg[1], step=drop_cfg[2])

        # 2. Pipeline Dataset Dinamis
        train_ds = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_TRAIN_FFB, image_size=self.cfg.IMG_SIZE, batch_size=bs, shuffle=True, label_mode='int', verbose=0
        ).prefetch(tf.data.AUTOTUNE)
        
        val_ds = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_VAL_FFB, image_size=self.cfg.IMG_SIZE, batch_size=bs, shuffle=False, label_mode='int', verbose=0
        ).prefetch(tf.data.AUTOTUNE)

        # 3. Graf Komputasi
        model = self._build_functional_graph(dropout_rate=drop)

        # 4. Injeksi Optimizer
        if opt_str == 'adamw':
            optimizer = optimizers.AdamW(learning_rate=lr, weight_decay=wd)
        elif opt_str == 'adam':
            optimizer = optimizers.Adam(learning_rate=lr, weight_decay=wd)
        else:
            optimizer = optimizers.SGD(learning_rate=lr, momentum=0.9, weight_decay=wd)

        model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

        # 5. Eksekusi Training (Hanya menyimpan log pruning, TIDAK menyimpan .keras)
        callbacks_list = [
            TFKerasPruningCallback(trial, "val_loss")
        ]

        history = model.fit(
            train_ds, validation_data=val_ds, epochs=self.cfg.EPOCHS_HPO,
            callbacks=callbacks_list, verbose=0
        )

        return min(history.history['val_loss'])

    # --------------------------------------------------------------------------
    # PUBLIC METHOD: THE GRAND LOOP
    # --------------------------------------------------------------------------
    def run_optimization(self) -> None:
        self.run_unit_test() 
        
        print(f"\n[INFO] Membuka koneksi SQLite di: {self.db_url}")
        study = optuna.create_study(
            study_name=self.study_name,
            storage=self.db_url,
            load_if_exists=True,
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=self.cfg.N_STARTUP_TRIALS, 
                n_warmup_steps=self.cfg.N_WARMUP_STEPS
            )
        )
        
        study.optimize(self._objective, n_trials=self.cfg.N_TRIALS)
        
        # ----------------------------------------------------------------------
        # EKSPOR BEST PARAMS KE JSON (Untuk digunakan di Notebook Training)
        # ----------------------------------------------------------------------
        best_params = study.best_params
        best_params['best_val_loss'] = study.best_value # Tambahkan metrik referensi
        
        with open(self.best_params_path, 'w') as f:
            json.dump(best_params, f, indent=4)
            
        print("\n" + "="*50)
        print(f"🏆 HASIL TERBAIK OPTUNA: {self.arch.value}")
        print("="*50)
        for key, value in best_params.items():
            if key != 'best_val_loss':
                print(f"  - {key.replace('_', ' ').title()}: {value}")
        print(f"  - Best Val Loss: {study.best_value:.4f}")
        print(f"  - File Ekspor JSON : {self.best_params_path}")

print("[BERHASIL] Cell 3 (HPO Engine) diperbarui. Auto-Save .keras Dihapus, diganti dengan JSON Exporter.")

[BERHASIL] Cell 3 (HPO Engine) diperbarui. Auto-Save .keras Dihapus, diganti dengan JSON Exporter.


/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `optuna.integration.tfkeras` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.tfkeras` instead.
  return _bootstrap._gcd_import(name[level:], package, level)


In [4]:
# ==============================================================================
# [CELL 4] EKSEKUSI THE GRAND LOOP (SINGLE/MULTI-ARCHITECTURE RUNNER)
# ==============================================================================
import gc
import tensorflow as tf

# Inisialisasi Konfigurasi Sentral
config = HPOConfig()

# ------------------------------------------------------------------------------
# 1. MODE SELEKSI ARSITEKTUR
# ------------------------------------------------------------------------------
# Silakan atur list ini. Anda bisa memasukkan 1, 2, atau ke-3 arsitektur sekaligus.
# Jika ingin mencoba 1 arsitektur saja, cukup tinggalkan satu item di dalam list.

MODELS_TO_TUNE = [
    ModelArch.MOBILENET_V3, 
    ModelArch.EFFICIENTNET_B0, 
    ModelArch.RESNET_50
]

print("\n" + "★"*75)
print(f"🔥 MEMULAI THE GRAND LOOP HPO UNTUK {len(MODELS_TO_TUNE)} ARSITEKTUR")
print("★"*75)

# ------------------------------------------------------------------------------
# 2. EKSEKUSI BERANTAI (AUTOMATED EXECUTION)
# ------------------------------------------------------------------------------
for idx, arch in enumerate(MODELS_TO_TUNE, start=1):
    print(f"\n{'='*75}")
    print(f"[{idx}/{len(MODELS_TO_TUNE)}] MENGINISIALISASI EKSPERIMEN: {arch.value}")
    print(f"{'='*75}")
    
    # A. Instansiasi Engine untuk Arsitektur Saat Ini
    engine = FFBHPOEngine(config=config, architecture=arch)
    
    # B. Eksekusi Pencarian Hyperparameter (Optuna + SQLite Logging)
    engine.run_optimization()
    
    # C. Protokol Pembersihan Memori Ekstrem (Wajib untuk Multi-Looping)
    # Menghapus instance engine dan membersihkan sisa graf komputasi di VRAM GPU
    print(f"\n[CLEANUP] Membersihkan alokasi memori VRAM GPU untuk {arch.value}...")
    del engine
    tf.keras.backend.clear_session()
    gc.collect()
    
    print(f"[{idx}/{len(MODELS_TO_TUNE)}] STATUS: {arch.value} SELESAI & AMAN.")

print("\n" + "★"*75)
print("🏆 SELURUH PROSES THE GRAND LOOP TELAH SELESAI DENGAN SUKSES!")
print("★"*75)

I0000 00:00:1784645384.648303      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🔥 MEMULAI THE GRAND LOOP HPO UNTUK 3 ARSITEKTUR
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

[1/3] MENGINISIALISASI EKSPERIMEN: MobileNetV3Small

🛠️ INISIALISASI BUILDER: MobileNetV3Small
[TEST] Menjalankan Sanity Check arsitektur...
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[TEST] LULUS! Output Shape: (1, 3) | Parameter: 943,155

[INFO] Membuka koneksi SQLite di: sqlite:////kaggle/working/hpo_outputs/MobileNetV3Small/study_MobileNetV3Small.db


[I 2026-07-21 14:49:54,419] A new study created in RDB with name: FFB_MobileNetV3Small_HPO
[I 2026-07-21 14:50:34,024] Trial 0 finished with value: 1.2034329175949097 and parameters: {'batch_size': 16, 'learning_rate': 0.0015751320499779737, 'weight_decay': 2.9380279387035334e-05, 'optimizer': 'sgd', 'dropout_rate': 0.2}. Best is trial 0 with value: 1.2034329175949097.
[I 2026-07-21 14:50:59,079] Trial 1 pruned. Trial was pruned at epoch 0.



🏆 HASIL TERBAIK OPTUNA: MobileNetV3Small
  - Batch Size: 16
  - Learning Rate: 0.0015751320499779737
  - Weight Decay: 2.9380279387035334e-05
  - Optimizer: sgd
  - Dropout Rate: 0.2
  - Best Val Loss: 1.2034
  - File Ekspor JSON : /kaggle/working/hpo_outputs/MobileNetV3Small/best_params_MobileNetV3Small.json

[CLEANUP] Membersihkan alokasi memori VRAM GPU untuk MobileNetV3Small...
[1/3] STATUS: MobileNetV3Small SELESAI & AMAN.

[2/3] MENGINISIALISASI EKSPERIMEN: EfficientNetB0

🛠️ INISIALISASI BUILDER: EfficientNetB0
[TEST] Menjalankan Sanity Check arsitektur...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[TEST] LULUS! Output Shape: (1, 3) | Parameter: 4,058,534

[INFO] Membuka koneksi SQLite di: sqlite:////kaggle/working/hpo_outputs/EfficientNetB0/study_EfficientNetB0.db


[I 2026-07-21 14:51:02,452] A new study created in RDB with name: FFB_EfficientNetB0_HPO
E0000 00:00:1784645474.328131      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/Edge_EfficientNetB0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
[I 2026-07-21 14:51:29,999] Trial 0 finished with value: 1.213018774986267 and parameters: {'batch_size': 16, 'learning_rate': 0.0015751320499779737, 'weight_decay': 2.9380279387035334e-05, 'optimizer': 'sgd', 'dropout_rate': 0.2}. Best is trial 0 with value: 1.213018774986267.
[I 2026-07-21 14:51:55,835] Trial 1 finished with value: 1.177802562713623 and parameters: {'batch_size': 32, 'learning_rate': 0.004622589001020831, 'weight_decay': 4.335281794951564e-05, 'optimizer': 'sgd', 'dropout_rate': 0.2}. Best is trial 1 with value: 1.177802562713623.



🏆 HASIL TERBAIK OPTUNA: EfficientNetB0
  - Batch Size: 32
  - Learning Rate: 0.004622589001020831
  - Weight Decay: 4.335281794951564e-05
  - Optimizer: sgd
  - Dropout Rate: 0.2
  - Best Val Loss: 1.1778
  - File Ekspor JSON : /kaggle/working/hpo_outputs/EfficientNetB0/best_params_EfficientNetB0.json

[CLEANUP] Membersihkan alokasi memori VRAM GPU untuk EfficientNetB0...
[2/3] STATUS: EfficientNetB0 SELESAI & AMAN.

[3/3] MENGINISIALISASI EKSPERIMEN: ResNet50

🛠️ INISIALISASI BUILDER: ResNet50
[TEST] Menjalankan Sanity Check arsitektur...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
[TEST] LULUS! Output Shape: (1, 3) | Parameter: 23,602,051

[INFO] Membuka koneksi SQLite di: sqlite:////kaggle/working/hpo_outputs/ResNet50/study_ResNet50.db


[I 2026-07-21 14:51:59,871] A new study created in RDB with name: FFB_ResNet50_HPO
[I 2026-07-21 14:52:28,121] Trial 0 finished with value: 1.520790457725525 and parameters: {'batch_size': 16, 'learning_rate': 0.0015751320499779737, 'weight_decay': 2.9380279387035334e-05, 'optimizer': 'sgd', 'dropout_rate': 0.2}. Best is trial 0 with value: 1.520790457725525.
[I 2026-07-21 14:52:53,946] Trial 1 finished with value: 1.4795247316360474 and parameters: {'batch_size': 32, 'learning_rate': 0.004622589001020831, 'weight_decay': 4.335281794951564e-05, 'optimizer': 'sgd', 'dropout_rate': 0.2}. Best is trial 1 with value: 1.4795247316360474.



🏆 HASIL TERBAIK OPTUNA: ResNet50
  - Batch Size: 32
  - Learning Rate: 0.004622589001020831
  - Weight Decay: 4.335281794951564e-05
  - Optimizer: sgd
  - Dropout Rate: 0.2
  - Best Val Loss: 1.4795
  - File Ekspor JSON : /kaggle/working/hpo_outputs/ResNet50/best_params_ResNet50.json

[CLEANUP] Membersihkan alokasi memori VRAM GPU untuk ResNet50...
[3/3] STATUS: ResNet50 SELESAI & AMAN.

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 SELURUH PROSES THE GRAND LOOP TELAH SELESAI DENGAN SUKSES!
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [5]:
# ==============================================================================
# [CELL 5] OOP VISUALIZER: ANALISIS HASIL HPO (INTERAKTIF & STATIS 4 PLOT)
# ==============================================================================
import os
import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink

class HPOVisualizer:
    """
    Standard 9: Base Class untuk memvisualisasikan hasil HPO dari database SQLite.
    Dirancang Agnostik (Standard 5) untuk menghasilkan 4 grafik statis standar Q1.
    """
    
    def __init__(self, config: HPOConfig, architecture: ModelArch):
        self.cfg = config
        self.arch = architecture
        self.study_name = f"FFB_{self.arch.value}_HPO"
        
        self.db_path = self.cfg.get_db_path(self.arch)
        self.db_url = f"sqlite:///{self.db_path}"
        self.base_output_dir = self.cfg.get_output_dir(self.arch)
        
        # Standard 4: Isolasi Struktur Direktori
        self.vis_dir = os.path.join(self.base_output_dir, "visualisation")
        os.makedirs(self.vis_dir, exist_ok=True)
        
        # Path Ekspor Statis (4 File)
        self.p_hist = os.path.join(self.vis_dir, f"hpo_1_{self.arch.value}_history.png")
        self.p_lr = os.path.join(self.vis_dir, f"hpo_2_{self.arch.value}_lr_vs_loss.png")
        self.p_batch = os.path.join(self.vis_dir, f"hpo_3_{self.arch.value}_batch_impact.png")
        self.p_opt = os.path.join(self.vis_dir, f"hpo_4_{self.arch.value}_opt_perf.png")
        
        print(f"\n" + "="*60)
        print(f"📊 INISIALISASI VISUALIZER: {self.arch.value}")
        print("="*60)
        
        self._validate_database_integrity()
        self.study = optuna.load_study(study_name=self.study_name, storage=self.db_url)

    def _validate_database_integrity(self) -> None:
        if not os.path.exists(self.db_path):
            raise FileNotFoundError(f"[TEST GAGAL] Database tidak ditemukan: {self.db_path}")
        if os.path.getsize(self.db_path) == 0:
            raise ValueError(f"[TEST GAGAL] Database kosong (0 bytes): {self.db_path}")

    def render_interactive_plots(self) -> None:
        """Menampilkan grafik interaktif bawaan Optuna di UI Notebook."""
        print("\n[INFO] Merender Grafik Interaktif Optuna...")
        try:
            display(optuna.visualization.plot_optimization_history(self.study))
            display(optuna.visualization.plot_param_importances(self.study))
            display(optuna.visualization.plot_parallel_coordinate(self.study))
        except Exception as e:
            print(f"[WARNING] Gagal merender grafik interaktif: {e}")

    def render_static_plots(self) -> None:
        """Mengekstrak DataFrame SQLite dan merender 4 Grafik Statis sesuai metodologi."""
        print("\n[INFO] Mengekstrak Data dari SQLite untuk 4 Render Statis...")
        
        df = self.study.trials_dataframe()
        df_completed = df[df['state'] == 'COMPLETE'].copy()
        
        if df_completed.empty:
            print("[WARNING] Tidak ada trial yang COMPLETE untuk divisualisasikan.")
            return

        # Mapping nama kolom Optuna ke format presentasi
        rename_map = {
            'value': 'Validation Loss',
            'number': 'Trial',
            'params_learning_rate': 'Learning Rate',
            'params_weight_decay': 'Weight Decay',
            'params_batch_size': 'Batch Size',
            'params_optimizer': 'Optimizer',
            'params_dropout_rate': 'Dropout Rate'
        }
        df_completed.rename(columns={k: v for k, v in rename_map.items() if k in df_completed.columns}, inplace=True)

        plt.style.use('default')
        sns.set_theme(style="ticks", context="notebook")

        # ----------------------------------------------------------------------
        # VISUALISASI 1: Optimization History
        # ----------------------------------------------------------------------
        plt.figure(figsize=(7, 5))
        plt.plot(df_completed['Trial'], df_completed['Validation Loss'], marker='o', 
                 linestyle='-', markersize=7, color='#1f77b4', linewidth=2)
        plt.title(f'Optimization History ({self.arch.value})', fontsize=14, pad=10)
        plt.xlabel('Trial', fontsize=12)
        plt.ylabel('Validation Loss', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(self.p_hist, dpi=300)
        plt.close()

        # ----------------------------------------------------------------------
        # VISUALISASI 2: Learning Rate vs Validation Loss
        # ----------------------------------------------------------------------
        plt.figure(figsize=(7, 5))
        plt.scatter(df_completed['Learning Rate'], df_completed['Validation Loss'], 
                    color='#1f77b4', s=50, alpha=0.9)
        plt.xscale('log')
        plt.title(f'LR vs Loss ({self.arch.value})', fontsize=14, pad=10)
        plt.xlabel('Learning Rate (Log Scale)', fontsize=12)
        plt.ylabel('Val Loss', fontsize=12)
        plt.grid(True, which="both", linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(self.p_lr, dpi=300)
        plt.close()

        # ----------------------------------------------------------------------
        # VISUALISASI 3: Batch Size Impact (Boxplot)
        # ----------------------------------------------------------------------
        plt.figure(figsize=(7, 5))
        if 'Batch Size' in df_completed.columns:
            sns.boxplot(x='Batch Size', y='Validation Loss', data=df_completed, color='white', 
                        linewidth=1.5, fliersize=5,
                        boxprops=dict(edgecolor='#1f77b4'),
                        whiskerprops=dict(color='#1f77b4'),
                        medianprops=dict(color='#2ecc71', linewidth=2))
            plt.title(f'Batch Size Impact ({self.arch.value})', fontsize=14, pad=10)
            plt.xlabel('Batch Size', fontsize=12)
            plt.ylabel('Val Loss', fontsize=12)
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(self.p_batch, dpi=300)
        plt.close()

        # ----------------------------------------------------------------------
        # VISUALISASI 4: Optimizer Performance (Min Loss)
        # ----------------------------------------------------------------------
        plt.figure(figsize=(6, 5))
        if 'Optimizer' in df_completed.columns:
            opt_perf = df_completed.groupby('Optimizer')['Validation Loss'].min().reset_index()
            sns.barplot(x='Optimizer', y='Validation Loss', data=opt_perf, color='#1f77b4', width=0.5)
            plt.title(f'Optimizer Performance ({self.arch.value})', fontsize=14, pad=10)
            plt.xlabel('Optimizer', fontsize=12)
            plt.ylabel('Minimum Val Loss', fontsize=12)
            plt.grid(axis='y', linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.savefig(self.p_opt, dpi=300)
        plt.close()

        print(f"[BERHASIL] 4 Gambar Resolusi Tinggi Tersimpan di: {self.vis_dir}")
        display(FileLink(self.p_hist))
        display(FileLink(self.p_lr))
        display(FileLink(self.p_batch))
        display(FileLink(self.p_opt))

print("[BERHASIL] Cell 5 (OOP Visualizer) diperbarui dengan 4 Plot Analitik.")

[BERHASIL] Cell 5 (OOP Visualizer) diperbarui dengan 4 Plot Analitik.


## Function Visualisation Execution

In [6]:
# ==============================================================================
# [CELL 6] EKSEKUSI RENDER VISUALISASI HPO (BATCH RUN)
# ==============================================================================

print(f"MEMULAI RENDER VISUALISASI UNTUK {len(MODELS_TO_TUNE)} ARSITEKTUR")

for arch in MODELS_TO_TUNE:
    try:
        visualizer = HPOVisualizer(config=config, architecture=arch)
        visualizer.render_interactive_plots()
        visualizer.render_static_plots()
        
    except FileNotFoundError as e:
        print(e)
    except Exception as e:
        print(f"[ERROR] Gagal memvisualisasikan {arch.value}: {e}")

print("Finish")

MEMULAI RENDER VISUALISASI UNTUK 3 ARSITEKTUR

📊 INISIALISASI VISUALIZER: MobileNetV3Small

[INFO] Merender Grafik Interaktif Optuna...


[WARNING] Gagal merender grafik interaktif: Cannot evaluate parameter importances with only a single trial.

[INFO] Mengekstrak Data dari SQLite untuk 4 Render Statis...
[BERHASIL] 4 Gambar Resolusi Tinggi Tersimpan di: /kaggle/working/hpo_outputs/MobileNetV3Small/visualisation


/kaggle/working/hpo_outputs/MobileNetV3Small/visualisation/hpo_1_MobileNetV3Small_history.png

/kaggle/working/hpo_outputs/MobileNetV3Small/visualisation/hpo_2_MobileNetV3Small_lr_vs_loss.png

/kaggle/working/hpo_outputs/MobileNetV3Small/visualisation/hpo_3_MobileNetV3Small_batch_impact.png

...

/kaggle/working/hpo_outputs/ResNet50/visualisation/hpo_1_ResNet50_history.png

/kaggle/working/hpo_outputs/ResNet50/visualisation/hpo_2_ResNet50_lr_vs_loss.png

/kaggle/working/hpo_outputs/ResNet50/visualisation/hpo_3_ResNet50_batch_impact.png

/kaggle/working/hpo_outputs/ResNet50/visualisation/hpo_4_ResNet50_opt_perf.png

Finish


In [7]:
import shutil
from IPython.display import FileLink, display

# Folder yang ingin didownload
folder_path = "/kaggle/working/hpo_outputs"

# Nama file ZIP output
zip_path = "/kaggle/working/hpo_outputs.zip"

# Buat ZIP
shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=folder_path
)

print(f"ZIP berhasil dibuat: {zip_path}")

# Tampilkan link download
display(FileLink(zip_path))

ZIP berhasil dibuat: /kaggle/working/hpo_outputs.zip


/kaggle/working/hpo_outputs.zip